# Reason Sandbox

> the single micro‑kernel entry point for inference, validation or ad‑hoc SPARQL CONSTRUCT. All provenance wrapping lives here.

In [ ]:
#| default_exp reason.sandbox

In [ ]:
#| export
from __future__ import annotations
import json
from typing import List, Tuple
from pydantic import BaseModel
from rdflib import Graph, RDF, Dataset
from cogitarelink.core.debug import get_logger
from cogitarelink.reason.prov import wrap_patch_with_prov
from pyshacl import validate

In [ ]:
#| export
log = get_logger("sandbox")


In [ ]:
#| export
def _to_graph(data:str|dict, fmt="json-ld") -> Dataset:
    g = Dataset()
    g.parse(data=data if isinstance(data,str) else json.dumps(data), format=fmt)
    return g


In [ ]:
#| export
def reason_over(*, 
                jsonld:str,
                shapes_turtle:str|None=None,
                query:str|None=None
               ) -> Tuple[str,str]:
    """
    • If `shapes_turtle` → run SHACL rules (iterate rules)  
    • Else if `query`    → run SPARQL CONSTRUCT  
    • Returns (patch_jsonld, nl_summary)
    """
    data=_to_graph(jsonld)
    patch = Graph()

    if shapes_turtle:
        shapes=_to_graph(shapes_turtle, fmt="turtle")
        conforms, r, results_graph = validate(data, shacl_graph=shapes,
                                  iterate_rules=True, advanced=True, inplace=True)
        # `data` mutated in‑place; diff = data − original
        patch += r
        
        # Check for violations more explicitly by checking the results_graph content
        violation_triples = []
        if results_graph is not None and len(results_graph) > 0:
            for s, p, o in results_graph:
                if 'ValidationResult' in str(o) or 'violation' in str(o).lower():
                    violation_triples.append((s, p, o))
        
        has_violations = len(violation_triples) > 0
        summary=f"SHACL run; conforms:{conforms}; added {len(r)} triples"
        
        if has_violations:
            summary += "; violations found"

    elif query:
        patch += data.query(query).graph
        summary=f"CONSTRUCT produced {len(patch)} triples"
    else:
        summary="no‑op"
    
    prov_patch = wrap_patch_with_prov(patch)
    return prov_patch.serialize(format="json-ld"), summary

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()